# Priority 1 — Gemma Reconstructed Full-Context Extraction

This notebook performs the frozen full-context condition for
`google/gemma-2-2b-it`.

Gemma-2's chat template did not support a separate system role in the
original generation notebook. The archived system instruction is therefore
folded into the user message exactly as it was during generation.

Responses are not regenerated. Prompt tokens are reconstructed, archived
completion text is re-encoded, and a teacher-forced forward pass extracts
the same 30-query × 250-key activation windows as the completion-only study.

In [2]:
!pip install -q "transformer_lens>=3.1,<4" transformers huggingface_hub

import gc
import hashlib
import json
import os
import platform
import shutil
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer

assert torch.cuda.is_available(), "Enable a Kaggle GPU before running."

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("TransformerLens:", version("transformer-lens"))

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 31.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.8 MB/s eta 0:00:0000:0100:01
GPU: Tesla T4
VRAM GB: 15.64
Python: 3.12.13
PyTorch: 2.10.0+cu128
Transformers: 5.16.1
TransformerLens: 3.8.0


## 1. Locate the final Phase-2 records

In [3]:
def find_unique_file(filenames, preferred_paths=()):
    for preferred in preferred_paths:
        p = Path(preferred)
        if p.exists():
            print("Using preferred input:", p)
            return p

    candidates = []
    for filename in filenames:
        for root in (Path("/kaggle/input"), Path("/kaggle/working")):
            if root.exists():
                candidates.extend(root.rglob(filename))

    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            "None of the requested files were found: "
            + ", ".join(filenames)
        )

    if len(candidates) > 1:
        print("Candidate files:")
        for i, p in enumerate(candidates):
            print(f"  [{i}] {p}")
        raise RuntimeError(
            "Multiple candidate label files were found. "
            "Set PHASE2_PATH manually to the intended final file."
        )

    print("Resolved input:", candidates[0])
    return candidates[0]


def load_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

PHASE2_PATH = find_unique_file(
    filenames=[
        "phase2_labeled_gemma_clean.jsonl",
        "phase2_labeled_gemma.jsonl",
    ],
    preferred_paths=[
        "/kaggle/input/notebooks/gunjalgupta/phase-2-gemma/pivot_research_gemma/phase2_labeled_gemma.jsonl",
        "/kaggle/working/phase2_labeled_gemma_clean.jsonl",
    ],
)

OUTPUT_DIR = Path("/kaggle/working/full_context_gemma")
RECORD_DIR = OUTPUT_DIR / "records"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RECORD_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = OUTPUT_DIR / "full_context_gemma_metadata.jsonl"
ALIGNMENT_PATH = OUTPUT_DIR / "full_context_gemma_alignment_audit.csv"
MANIFEST_PATH = OUTPUT_DIR / "full_context_gemma_manifest.json"
ERRORS_PATH = OUTPUT_DIR / "full_context_gemma_errors.jsonl"

all_records = load_jsonl(PHASE2_PATH)
usable_records = [
    r for r in all_records
    if r.get("label") in {"correction", "hallucination_pivot"}
    and int(r.get("pivot_token_start", -1)) >= 30
]

counts = Counter(r["label"] for r in usable_records)
print("All records:", len(all_records))
print("Usable:", len(usable_records))
print("Labels:", dict(counts))

assert len(all_records) == 108
assert len(usable_records) == 76
assert counts["correction"] == 13
assert counts["hallucination_pivot"] == 63

Using preferred input: /kaggle/input/notebooks/gunjalgupta/phase-2-gemma/pivot_research_gemma/phase2_labeled_gemma.jsonl
All records: 108
Usable: 76
Labels: {'hallucination_pivot': 63, 'correction': 13}


## 2. Load Gemma with the same Phase-3 extraction stack

In [4]:
MODEL_NAME = "google/gemma-2-2b-it"
MODEL_DTYPE = torch.bfloat16

gc.collect()
torch.cuda.empty_cache()

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_NAME,
    device="cuda",
    dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
model.eval()
tokenizer = model.tokenizer

N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads
D_HEAD = model.cfg.d_head
D_MODEL = model.cfg.d_model

print("Layers:", N_LAYERS)
print("Heads:", N_HEADS)
print("d_head:", D_HEAD)
print("d_model:", D_MODEL)
print("Tokenizer:", type(tokenizer).__name__)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
Layers: 26
Heads: 8
d_head: 256
d_model: 2304
Tokenizer: GemmaTokenizer


## 3. Freeze exact prompt reconstruction

In [5]:
SYSTEM_PROMPT = 'You are a careful, honest reasoner. When solving a problem:\n Think through the problem step by step in plain prose, working toward an answer.\n\nIf, partway through, you realize your reasoning so far contains an actual error —\nnot because you were told to check, but because you have spotted something\nconcretely wrong — stop and correct it before continuing. If your reasoning is\nalready correct, continue straight to the final answer without pausing to\nannounce that you checked it.\n\nDo not narrate a "checking," "pause," or "review" step unless you have genuinely\nfound something to fix. Do not use headers, bold text, or numbered steps to\nstructure your answer — just write continuous prose.'
USER_SUFFIX = (
    "\n\nRemember: commit to a first answer, then double-check it. "
    "Show your doubt explicitly."
)

def build_serialized_prompt(question):
    combined = SYSTEM_PROMPT + "\n\n" + question + USER_SUFFIX
    messages = [{"role": "user", "content": combined}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def encode_generation_prompt(question):
    serialized = build_serialized_prompt(question)

    raw_ids = tokenizer(
        serialized,
        add_special_tokens=True,
        truncation=False,
        return_tensors="pt",
    )["input_ids"][0]

    used_ids = tokenizer(
        serialized,
        add_special_tokens=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )["input_ids"][0]

    return {
        "serialized": serialized,
        "raw_ids": raw_ids,
        "used_ids": used_ids,
        "was_truncated": len(raw_ids) != len(used_ids),
    }


print(build_serialized_prompt("Test question")[:800])
print("System SHA256:", sha256_text(SYSTEM_PROMPT))
print("Chat template SHA256:", sha256_text(tokenizer.chat_template or ""))

<bos><start_of_turn>user
You are a careful, honest reasoner. When solving a problem:
 Think through the problem step by step in plain prose, working toward an answer.

If, partway through, you realize your reasoning so far contains an actual error —
not because you were told to check, but because you have spotted something
concretely wrong — stop and correct it before continuing. If your reasoning is
already correct, continue straight to the final answer without pausing to
announce that you checked it.

Do not narrate a "checking," "pause," or "review" step unless you have genuinely
found something to fix. Do not use headers, bold text, or numbered steps to
structure your answer — just write continuous prose.

Test question

Remember: commit to a first answer, then double-check it. Show yo
System SHA256: 22a092752b16fbf226da933564d22a0a2502f66ad4fd99027048c28a6d69bbb7
Chat template SHA256: ecd6ae513fe103f0eb62e8ab5bfa8d0fe45c1074fa398b089c93a7e70c15cfd6


## 4. Alignment audit

In [6]:
QUERY_WINDOW_SIZE = 30
MAX_KEY_CONTEXT = 250

alignment_rows = []
prepared = []

for source_idx, r in enumerate(usable_records):
    prompt = encode_generation_prompt(r["prompt_text"])
    prompt_ids = prompt["used_ids"].to(dtype=torch.long)

    completion_ids = tokenizer.encode(
        r["generated_text"],
        add_special_tokens=False,
        return_tensors="pt",
    )[0].to(dtype=torch.long)

    pivot_completion_idx = int(r["pivot_token_start"])
    pivot_full_idx = len(prompt_ids) + pivot_completion_idx
    full_ids = torch.cat([prompt_ids, completion_ids], dim=0)

    query_completion = completion_ids[
        pivot_completion_idx - QUERY_WINDOW_SIZE:
        pivot_completion_idx
    ]
    query_full = full_ids[
        pivot_full_idx - QUERY_WINDOW_SIZE:
        pivot_full_idx
    ]

    query_exact = bool(torch.equal(query_completion, query_full))
    pivot_valid = (
        pivot_completion_idx >= QUERY_WINDOW_SIZE
        and pivot_completion_idx < len(completion_ids)
    )

    neighborhood = tokenizer.decode(
        completion_ids[
            max(0, pivot_completion_idx - 6):
            min(len(completion_ids), pivot_completion_idx + 10)
        ]
    )

    key_start = max(0, pivot_full_idx - MAX_KEY_CONTEXT)
    key_positions = np.arange(key_start, pivot_full_idx)
    prompt_keys = int(np.sum(key_positions < len(prompt_ids)))
    completion_keys = int(len(key_positions) - prompt_keys)

    total_tokens_recorded = r.get("total_tokens")
    total_tokens_match = (
        total_tokens_recorded is None
        or int(total_tokens_recorded) == len(completion_ids)
    )

    key = f'{r["prompt_id"]}#{r.get("sample_idx")}'
    row = {
        "source_idx": source_idx,
        "key": key,
        "prompt_id": r["prompt_id"],
        "sample_idx": r.get("sample_idx"),
        "label": r["label"],
        "prompt_tokens_raw": int(len(prompt["raw_ids"])),
        "prompt_tokens_used": int(len(prompt_ids)),
        "prompt_was_truncated": bool(prompt["was_truncated"]),
        "completion_tokens_reencoded": int(len(completion_ids)),
        "completion_tokens_recorded": (
            None if total_tokens_recorded is None
            else int(total_tokens_recorded)
        ),
        "completion_length_matches_record": bool(total_tokens_match),
        "pivot_completion_idx": pivot_completion_idx,
        "pivot_full_idx": int(pivot_full_idx),
        "pivot_valid": bool(pivot_valid),
        "query_token_ids_exact": bool(query_exact),
        "prompt_keys_in_250_window": prompt_keys,
        "completion_keys_in_250_window": completion_keys,
        "pivot_neighborhood": neighborhood,
    }
    alignment_rows.append(row)

    prepared.append({
        "source_idx": source_idx,
        "record": r,
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "full_ids": full_ids,
        "pivot_full_idx": pivot_full_idx,
        "prompt_info": prompt,
        "alignment": row,
    })

alignment_df = pd.DataFrame(alignment_rows)
alignment_df.to_csv(ALIGNMENT_PATH, index=False)

print(alignment_df[
    [
        "prompt_was_truncated",
        "completion_length_matches_record",
        "pivot_valid",
        "query_token_ids_exact",
    ]
].apply(pd.Series.value_counts).fillna(0))

assert alignment_df["pivot_valid"].all()
assert alignment_df["query_token_ids_exact"].all()

failure_rate = 1.0 - alignment_df[
    "completion_length_matches_record"
].mean()
print("Recorded completion-length mismatch rate:", f"{failure_rate:.1%}")

if failure_rate > 0.05:
    raise RuntimeError(
        "More than 5% of completions fail the archived token-length check. "
        "Stop and investigate before extraction."
    )

print("Alignment audit saved:", ALIGNMENT_PATH)

       prompt_was_truncated  completion_length_matches_record  pivot_valid  \
False                  76.0                               0.0          0.0   
True                    0.0                              76.0         76.0   

       query_token_ids_exact  
False                    0.0  
True                    76.0  
Recorded completion-length mismatch rate: 0.0%
Alignment audit saved: /kaggle/working/full_context_gemma/full_context_gemma_alignment_audit.csv


## 5. Low-memory full-context extraction

In [7]:
def extract_full_context(prepared_record):
    full_ids = prepared_record["full_ids"].unsqueeze(0).to("cuda")
    pivot_idx = int(prepared_record["pivot_full_idx"])

    query_start = pivot_idx - QUERY_WINDOW_SIZE
    query_end = pivot_idx
    key_real_start = max(0, pivot_idx - MAX_KEY_CONTEXT)
    key_end = pivot_idx

    n_real_keys = key_end - key_real_start
    valid_key_start = MAX_KEY_CONTEXT - n_real_keys

    weights_by_layer = [None] * N_LAYERS
    outputs_by_layer = [None] * N_LAYERS

    def make_pattern_hook(layer_idx):
        def hook_fn(pattern, hook):
            sliced = pattern[
                0,
                :,
                query_start:query_end,
                key_real_start:key_end,
            ]
            sliced_cpu = (
                sliced.detach()
                .to("cpu", dtype=torch.float32)
                .numpy()
            )
            padded = np.zeros(
                (N_HEADS, QUERY_WINDOW_SIZE, MAX_KEY_CONTEXT),
                dtype=np.float32,
            )
            padded[:, :, valid_key_start:] = sliced_cpu
            weights_by_layer[layer_idx] = padded
            return None
        return hook_fn

    def make_z_hook(layer_idx):
        def hook_fn(z, hook):
            sliced = z[0, query_start:query_end, :, :]
            outputs_by_layer[layer_idx] = (
                sliced.detach()
                .to("cpu", dtype=torch.float32)
                .numpy()
            )
            return None
        return hook_fn

    hooks = []
    for layer in range(N_LAYERS):
        hooks.append((
            f"blocks.{layer}.attn.hook_pattern",
            make_pattern_hook(layer),
        ))
        hooks.append((
            f"blocks.{layer}.attn.hook_z",
            make_z_hook(layer),
        ))

    with torch.inference_mode():
        model.run_with_hooks(
            full_ids,
            fwd_hooks=hooks,
            return_type=None,
        )

    missing_weights = [
        i for i, value in enumerate(weights_by_layer)
        if value is None
    ]
    missing_outputs = [
        i for i, value in enumerate(outputs_by_layer)
        if value is None
    ]
    if missing_weights or missing_outputs:
        raise RuntimeError(
            f"Missing hooks: weights={missing_weights}, "
            f"outputs={missing_outputs}"
        )

    weights = np.stack(weights_by_layer, axis=0)
    outputs = np.stack(outputs_by_layer, axis=0)

    del full_ids, weights_by_layer, outputs_by_layer, hooks
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "weights": weights,
        "outputs": outputs,
        "valid_key_start": valid_key_start,
        "n_real_keys": n_real_keys,
        "full_seq_len": int(len(prepared_record["full_ids"])),
    }

## 6. Mandatory one-record dry run

In [8]:
test_item = prepared[0]
print(
    test_item["record"]["prompt_id"],
    test_item["record"].get("sample_idx"),
    test_item["record"]["label"],
)

start = time.time()
test_result = extract_full_context(test_item)
elapsed = time.time() - start

assert test_result["weights"].shape == (
    N_LAYERS,
    N_HEADS,
    QUERY_WINDOW_SIZE,
    MAX_KEY_CONTEXT,
)
assert test_result["outputs"].shape == (
    N_LAYERS,
    QUERY_WINDOW_SIZE,
    N_HEADS,
    D_HEAD,
)

print("Elapsed seconds:", round(elapsed, 2))
print("Weights:", test_result["weights"].shape)
print("Outputs:", test_result["outputs"].shape)
print("Full sequence length:", test_result["full_seq_len"])
print("Real keys:", test_result["n_real_keys"])

del test_result
gc.collect()
torch.cuda.empty_cache()

mis_04 4 hallucination_pivot
Elapsed seconds: 1.69
Weights: (26, 8, 30, 250)
Outputs: (26, 30, 8, 256)
Full sequence length: 448
Real keys: 250


## 7. Extract all 76 records

In [9]:
RESET_OUTPUTS = True

if RESET_OUTPUTS and RECORD_DIR.exists():
    shutil.rmtree(RECORD_DIR)
    RECORD_DIR.mkdir(parents=True, exist_ok=True)

metadata = []
errors = []
start_time = time.time()

for item in tqdm(prepared, desc="Gemma full-context extraction"):
    r = item["record"]
    try:
        result = extract_full_context(item)
        out_idx = len(metadata)
        record_file = RECORD_DIR / f"record_{out_idx:03d}.npz"

        np.savez_compressed(
            record_file,
            weights=result["weights"],
            outputs=result["outputs"],
        )

        alignment = item["alignment"]
        metadata.append({
            "idx": out_idx,
            "source_idx": item["source_idx"],
            "record_file": record_file.name,
            "prompt_id": r["prompt_id"],
            "sample_idx": r.get("sample_idx"),
            "category": r.get("category"),
            "label": r["label"],
            "label_method": r.get("label_method"),
            "human_verified": r.get("human_verified", False),
            "pivot_type": r.get("pivot_type"),
            "pivot_phrase": r.get("pivot_phrase"),
            "pivot_completion_idx": int(r["pivot_token_start"]),
            "pivot_full_idx": int(item["pivot_full_idx"]),
            "prompt_token_count": int(len(item["prompt_ids"])),
            "completion_token_count": int(len(item["completion_ids"])),
            "full_seq_len": int(result["full_seq_len"]),
            "query_window_size": QUERY_WINDOW_SIZE,
            "max_key_context": MAX_KEY_CONTEXT,
            "valid_key_start": int(result["valid_key_start"]),
            "n_real_keys": int(result["n_real_keys"]),
            "prompt_keys_in_window": int(
                alignment["prompt_keys_in_250_window"]
            ),
            "completion_keys_in_window": int(
                alignment["completion_keys_in_250_window"]
            ),
            "prompt_was_truncated": bool(
                alignment["prompt_was_truncated"]
            ),
            "completion_length_matches_record": bool(
                alignment["completion_length_matches_record"]
            ),
            "model_name": MODEL_NAME,
            "condition": "reconstructed_full_context",
        })

        del result
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as exc:
        errors.append({
            "source_idx": item["source_idx"],
            "prompt_id": r.get("prompt_id"),
            "sample_idx": r.get("sample_idx"),
            "label": r.get("label"),
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        print(
            "ERROR",
            r.get("prompt_id"),
            r.get("sample_idx"),
            type(exc).__name__,
            str(exc),
        )

elapsed = time.time() - start_time

with METADATA_PATH.open("w", encoding="utf-8") as f:
    for row in metadata:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with ERRORS_PATH.open("w", encoding="utf-8") as f:
    for row in errors:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Successful:", len(metadata))
print("Errors:", len(errors))
print("Elapsed minutes:", round(elapsed / 60, 2))

assert len(metadata) == 76
assert not errors

Gemma full-context extraction:   0%|          | 0/76 [00:00<?, ?it/s]

Successful: 76
Errors: 0
Elapsed minutes: 4.3


## 8. Save manifest and validate outputs

In [10]:
tokenizer_commit = getattr(tokenizer, "init_kwargs", {}).get(
    "_commit_hash"
)

manifest = {
    "study_condition": "reconstructed_full_context",
    "model_name": MODEL_NAME,
    "model_dtype": str(MODEL_DTYPE),
    "model_revision_pinned_in_original_study": False,
    "tokenizer_commit_observed_now": tokenizer_commit,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": version("transformers"),
    "transformer_lens_version": version("transformer-lens"),
    "tokenizer_class": type(tokenizer).__name__,
    "system_prompt_sha256": sha256_text(SYSTEM_PROMPT),
    "chat_template_sha256": sha256_text(
        tokenizer.chat_template or ""
    ),
    "phase2_path": str(PHASE2_PATH),
    "n_phase2_records": len(all_records),
    "n_requested": len(usable_records),
    "n_extracted": len(metadata),
    "n_errors": len(errors),
    "query_window_size": QUERY_WINDOW_SIZE,
    "max_key_context": MAX_KEY_CONTEXT,
    "label_counts": dict(Counter(m["label"] for m in metadata)),
    "prompt_truncation_count": int(
        sum(m["prompt_was_truncated"] for m in metadata)
    ),
    "completion_length_mismatch_count": int(
        sum(
            not m["completion_length_matches_record"]
            for m in metadata
        )
    ),
    "elapsed_minutes": elapsed / 60,
    "reconstruction_note": (
        "Gemma system instructions were folded into the user message, "
        "matching the archived generation notebook. Original generated "
        "token IDs were not saved, so completion IDs were re-encoded from "
        "archived generated_text."
    ),
}

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

for m in tqdm(metadata, desc="Validating saved tensors"):
    with np.load(RECORD_DIR / m["record_file"]) as archive:
        assert archive["weights"].shape == (
            N_LAYERS,
            N_HEADS,
            QUERY_WINDOW_SIZE,
            MAX_KEY_CONTEXT,
        )
        assert archive["outputs"].shape == (
            N_LAYERS,
            QUERY_WINDOW_SIZE,
            N_HEADS,
            D_HEAD,
        )

print("READY FOR PAIRED COMPARISON")
print("Records:", RECORD_DIR)
print("Metadata:", METADATA_PATH)
print("Alignment audit:", ALIGNMENT_PATH)
print("Manifest:", MANIFEST_PATH)

Validating saved tensors:   0%|          | 0/76 [00:00<?, ?it/s]

READY FOR PAIRED COMPARISON
Records: /kaggle/working/full_context_gemma/records
Metadata: /kaggle/working/full_context_gemma/full_context_gemma_metadata.jsonl
Alignment audit: /kaggle/working/full_context_gemma/full_context_gemma_alignment_audit.csv
Manifest: /kaggle/working/full_context_gemma/full_context_gemma_manifest.json


In [11]:
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED
from IPython.display import FileLink, display

source_dir = Path("/kaggle/working/full_context_gemma")
zip_path = Path("/kaggle/working/full_context_gemma_complete.zip")

if not source_dir.exists():
    raise FileNotFoundError(f"Folder does not exist: {source_dir}")

files = sorted(p for p in source_dir.rglob("*") if p.is_file())
npz_files = [p for p in files if p.suffix == ".npz"]

print(f"Source folder: {source_dir}")
print(f"Total files found: {len(files)}")
print(f"NPZ files found: {len(npz_files)}")

if len(npz_files) != 76:
    raise RuntimeError(
        f"Expected 76 NPZ record files, but found {len(npz_files)}."
    )

# Remove any incomplete ZIP from a previous attempt.
if zip_path.exists():
    zip_path.unlink()

print("\nCreating ZIP...")

with ZipFile(zip_path, mode="w", compression=ZIP_STORED) as archive:
    for index, file_path in enumerate(files, start=1):
        # Preserve full_context_gemma/... inside the archive.
        archive_name = file_path.relative_to(source_dir.parent)
        archive.write(file_path, arcname=archive_name)

        if index % 20 == 0 or index == len(files):
            print(f"Archived {index}/{len(files)} files")

# Verify the archive.
with ZipFile(zip_path, "r") as archive:
    corrupted_member = archive.testzip()
    archived_npz = [
        name for name in archive.namelist()
        if name.endswith(".npz")
    ]

if corrupted_member is not None:
    raise RuntimeError(f"Corrupted ZIP entry: {corrupted_member}")

if len(archived_npz) != 76:
    raise RuntimeError(
        f"ZIP contains {len(archived_npz)} NPZ files instead of 76."
    )

print("\nZIP successfully created.")
print(f"Path: {zip_path}")
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MiB")
print(f"NPZ files inside ZIP: {len(archived_npz)}")
print("Integrity check: PASS")

display(FileLink(str(zip_path)))

Source folder: /kaggle/working/full_context_gemma
Total files found: 80
NPZ files found: 76

Creating ZIP...
Archived 20/80 files
Archived 40/80 files
Archived 60/80 files
Archived 80/80 files

ZIP successfully created.
Path: /kaggle/working/full_context_gemma_complete.zip
Size: 415.88 MiB
NPZ files inside ZIP: 76
Integrity check: PASS


/kaggle/working/full_context_gemma_complete.zip